# Tutorial 50 — Structure source files into chunks

[Tutorial 40](./40_repository_files.ipynb) selected repository paths without
reading their contents. This tutorial takes the next step: it reads one QAChat
source file, identifies its natural structural units, and turns those units
into searchable chunks.

The key distinction is between a **source unit**, which follows the structure of
the original file, and a **chunk**, which is the bounded piece of text passed to
retrieval. Small source units stay whole; larger units are divided while keeping
source location and symbol metadata.

[Tutorial 60](./60_embeddings.ipynb) will encode these chunks as embedding
vectors.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Read one selected file

Here source means the text inside a file that already passed repository
selection. We use `qachat/repository.py` so every following source unit and line
range can be compared with real QAChat code.


In [ ]:
SOURCE_PATH = "qachat/repository.py"


In [ ]:
report = qachat.scan_repository(
    root,
    extensions=[".py"],
    include_paths=[SOURCE_PATH],
)

print(report["selected_files"])


In [ ]:
source_file = qachat.read_repository_files(report)[0]

print(f"Path: {source_file.path}")
print(f"Lines: {len(source_file.text.splitlines())}")


## 3. Identify natural source units

A natural source unit follows the structure of the original file rather than an
arbitrary text window. For Python, QAChat can preserve module context and
complete functions, classes, and methods. The parser reads syntax and metadata;
it does not execute the source code.


In [ ]:
units = qachat.extract_source_units([source_file])

for unit in units:
    print(unit)


In [ ]:
print(f"Natural units: {len(units)}")


In [ ]:
function_units = [
    unit
    for unit in units
    if unit.kind == "python_function"
]

print(f"Function units: {len(function_units)}")


In [ ]:
def count_words(text: str) -> int:
    return len(text.split())


In [ ]:
for unit in units:
    location = f"{unit.start_line}-{unit.end_line}"
    size = count_words(unit.text)
    print(f"{location:<9} {size:>3} words ({unit.kind})   {unit.symbol}")


The line ranges come from the actual file. Each function unit starts at its
`def` line and ends with that complete function.


In [ ]:
example_unit = min(
    function_units,
    key=lambda unit: count_words(unit.text),
)

print(f"Example natural unit: {example_unit.symbol}")


In [ ]:
print(example_unit.text)


## 4. Turn source units into chunks

Retrieval searches bounded chunks. A natural unit that fits below the maximum
size remains intact. A larger unit is divided into overlapping chunks, and
every resulting chunk retains its source path, symbol, kind, and line ranges.

Smaller chunks can improve retrieval precision, while larger chunks preserve
more context. The small word-based limits below are intentionally chosen to make
the split visible. Production indexing uses the embedding model's token counter
instead of this teaching-only word counter.


In [ ]:
TARGET_WORDS = 120
MAX_WORDS = 160
OVERLAP_WORDS = 20


In [ ]:
chunks = qachat.chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=TARGET_WORDS,
    max_tokens=MAX_WORDS,
    overlap_tokens=OVERLAP_WORDS,
)

print(f"Natural units: {len(units)}")
print(f"Searchable chunks: {len(chunks)}")


In [ ]:
for chunk in chunks:
    print(chunk)


### A complete function chunk

The shortest real function fits below the maximum, so its natural unit and its chunk are identical.


In [ ]:
whole_unit = min(
    function_units,
    key=lambda unit: count_words(unit.text),
)
whole_chunks = [
    chunk
    for chunk in chunks
    if chunk.symbol == whole_unit.symbol
]

assert len(whole_chunks) == 1
assert whole_chunks[0].text == whole_unit.text

print(f"Function: {whole_unit.symbol}")


In [ ]:
print(f"Natural unit: {whole_unit.start_line}-{whole_unit.end_line}\n")
print(whole_unit.text)


In [ ]:
print(f"Chunk: {whole_chunks[0].start_line}-{whole_chunks[0].end_line}\n")
print(whole_chunks[0].text)


### Chunks from one larger function

Now select the smallest function that exceeds the maximum. Its chunks cover only parts of the function. Overlapping line ranges show where a little boundary context was repeated.


In [ ]:
oversized_units = [
    unit
    for unit in function_units
    if count_words(unit.text) > MAX_WORDS
]
split_unit = min(
    oversized_units,
    key=lambda unit: count_words(unit.text),
)
split_chunks = [
    chunk
    for chunk in chunks
    if chunk.symbol == split_unit.symbol
]

assert len(split_chunks) > 1

print(f"Function: {split_unit.symbol}")


In [ ]:
print(f"Natural unit: {split_unit.start_line}-{split_unit.end_line}\n")
print(split_unit.text)


In [ ]:
for number, chunk in enumerate(split_chunks, start=1):
    starts_at_def = chunk.text.lstrip().startswith("def ")
    print(
        f"Chunk {number} (Symbol: {chunk.symbol}): "
        f"lines {chunk.start_line}-{chunk.end_line}, "
        f"{count_words(chunk.text)} words, "
        f"starts at def: {starts_at_def}\n"
    )
    print(chunk.text)


Every chunk still carries the original function symbol, so later search results can point back to the correct function and lines.


## Continue the tutorial series

**Previous:** [Tutorial 40 — Select repository files](./40_repository_files.ipynb)

**Next:** [Tutorial 60 — Create embeddings](./60_embeddings.ipynb)
